# 01. Penn World Tableの取得

Penn World Table 11.0（185か国・1950〜2023年）から、家計消費の物価水準(`pl_con`)と
購買力平価ベース一人当たりGDP(`rgdpe_pc`)を取得し、各年の米国＝1.00に換算した
相対物価水準・相対所得を作る。

PWT 11.0のファイル（`pwt110.xlsx`）は`data/raw/pwt110_raw.xlsx`として保存する。
すでにこのファイルがあればそれを使い、なければDataverseNLからダウンロードする。
自動ダウンロードができない場合は、DOI 10.34894/FABVLR のページから`pwt110.xlsx`を
手動で入手し、`data/raw/pwt110_raw.xlsx`という名前で保存してから実行する。


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
for d in (RAW_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"raw: {(RAW_DIR).relative_to(BASE_DIR)}")
print(f"processed: {(PROCESSED_DIR).relative_to(BASE_DIR)}")

raw: data\raw
processed: data\processed


## 1. Penn World Table 11.0 を取得し、価格水準・所得を「各年の米国＝1.00」に換算

PWT 11.0はICP 2021を取り込んでおり、価格・実質GDPの基準年は2017年から2021年へ移っている。
ここでは各年の米国の`pl_con`・`rgdpe_pc`で割り直すため、基準年そのものには依存しない。

In [2]:
PWT_URL = "https://dataverse.nl/api/access/datafile/554105"  # pwt110.xlsx

PWT_FILE = RAW_DIR / "pwt110_raw.xlsx"
if not PWT_FILE.exists():
    response = requests.get(PWT_URL, timeout=180)
    response.raise_for_status()
    PWT_FILE.write_bytes(response.content)

pwt = pd.read_excel(RAW_DIR / "pwt110_raw.xlsx", sheet_name="Data")
pwt["rgdpe_pc"] = pwt["rgdpe"] / pwt["pop"]
pwt["cgdpe_pc"] = pwt["cgdpe"] / pwt["pop"]
# rgdpe（chained PPP）は年をまたぐ比較に、cgdpe（current PPP）は単年の横断面比較に
# 適しているとPWT公式ドキュメントが案内している。
# 本稿は主に年をまたぐパネル回帰にrgdpeを使うが、単年横断面（2023年）の頑健性確認用に
# cgdpeベースの相対所得も併せて構築する。

us = pwt[pwt["countrycode"] == "USA"].set_index("year")
pwt["relp"] = pwt["pl_con"] / pwt["year"].map(us["pl_con"])         # 対米・家計消費価格水準
pwt["reli"] = pwt["rgdpe_pc"] / pwt["year"].map(us["rgdpe_pc"])     # 対米・一人当たりGDP（chained PPP）
pwt["reli_c"] = pwt["cgdpe_pc"] / pwt["year"].map(us["cgdpe_pc"])   # 対米・一人当たりGDP（current PPP）

print(f"PWT: {pwt.shape[0]:,}行、{pwt['countrycode'].nunique()}か国、{pwt['year'].min()}〜{pwt['year'].max()}年")
pwt[pwt["countrycode"].isin(["JPN", "KOR", "TWN", "USA", "DEU", "GBR"])][
    ["countrycode", "year", "pl_con", "rgdpe_pc", "relp", "reli"]
].query("year == 2023")

PWT: 13,690行、185か国、1950〜2023年


,countrycode,year,pl_con,rgdpe_pc,relp,reli
3477,DEU,2023,0.918093,59853.647322,0.836963,0.801751
4587,GBR,2023,0.994785,52507.479948,0.906877,0.703348
6511,JPN,2023,0.790653,43363.668365,0.720785,0.580864
6955,KOR,2023,0.720639,52388.788643,0.656957,0.701758
12653,TWN,2023,0.485919,65004.351926,0.442980,0.870745
13023,USA,2023,1.096934,74653.676930,1.000000,1.000000


## 2. 対象国・対象年の確認

台湾（TWN）がWorld Bankにはないが、PWTには収録されていることを確認する。

In [3]:
has_taiwan = "TWN" in pwt["countrycode"].unique()
print(f"台湾（TWN）はPWTに収録されているか: {has_taiwan}")

coverage = (
    pwt.dropna(subset=["relp", "reli"])
    .groupby("countrycode")["year"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "first_year", "max": "last_year", "count": "n_years"})
)
print(f"relp・reliが両方そろう国数: {len(coverage)}")
coverage.loc[["JPN", "KOR", "TWN", "USA"]]

台湾（TWN）はPWTに収録されているか: True
relp・reliが両方そろう国数: 185


,first_year,last_year,n_years
countrycode,,,
JPN,1950,2023,74
KOR,1953,2023,71
TWN,1951,2023,73
USA,1950,2023,74


## 3. 保存

In [4]:
out_cols = ["countrycode", "country", "year", "pl_con", "rgdpe_pc", "cgdpe_pc", "pop", "relp", "reli", "reli_c"]
pwt_clean = pwt[out_cols].dropna(subset=["relp", "reli"]).copy()
pwt_clean.to_csv(PROCESSED_DIR / "pwt_relative_price_income.csv", index=False)
print(f"保存先: {(PROCESSED_DIR / 'pwt_relative_price_income.csv').relative_to(BASE_DIR)}")
print(f"行数: {len(pwt_clean):,}、国数: {pwt_clean['countrycode'].nunique()}")

保存先: data\processed\pwt_relative_price_income.csv
行数: 11,201、国数: 185


## まとめ

`data/processed/pwt_relative_price_income.csv` に、国・年別の対米相対物価水準(`relp`)・
対米相対所得(`reli`)を保存した。

次は `02_ppp_deviation_panel.ipynb` で、reverse B-S回帰に使う多国間パネルを整形する。